In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('/opt/workspace')\n",
    "\n",
    "from datetime import datetime\n",
    "from connectors.clickhouse_client import ClickHouseClient"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ref_date = datetime.now().strftime(\"%Y-%m-%d\")\n",
    "ref_date"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client = ClickHouseClient()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "change_metrics_query = f\"\"\"\n",
    "INSERT INTO gold.daily_change_metrics (\n",
    "    ref_date, table_name, total_inserts, total_updates,\n",
    "    total_deletes, total_active_records\n",
    ")\n",
    "SELECT\n",
    "    ref_date,\n",
    "    table_name,\n",
    "    countIf(operation_type = 'INSERT') AS total_inserts,\n",
    "    countIf(operation_type = 'UPDATE') AS total_updates,\n",
    "    countIf(operation_type = 'DELETE') AS total_deletes,\n",
    "    (\n",
    "        SELECT count()\n",
    "        FROM silver.current_state AS cs\n",
    "        WHERE cs.table_name = de.table_name\n",
    "            AND cs.is_active = 1\n",
    "    ) AS total_active_records\n",
    "FROM silver.delta_events AS de\n",
    "WHERE ref_date = toDate('{ref_date}')\n",
    "GROUP BY ref_date, table_name\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(change_metrics_query)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "change_metrics_result = client.execute_query_with_result(\n",
    "    f\"\"\"\n",
    "    SELECT *\n",
    "    FROM gold.daily_change_metrics\n",
    "    WHERE ref_date = toDate('{ref_date}')\n",
    "    \"\"\"\n",
    ")\n",
    "change_metrics_result.result_rows"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "quality_metrics_query = f\"\"\"\n",
    "INSERT INTO gold.data_quality_metrics (\n",
    "    ref_date, table_name, null_count, duplicate_count,\n",
    "    total_records, quality_score\n",
    ")\n",
    "SELECT\n",
    "    toDate('{ref_date}') AS ref_date,\n",
    "    table_name,\n",
    "    countIf(data = '' OR data IS NULL) AS null_count,\n",
    "    count() - uniq(primary_key) AS duplicate_count,\n",
    "    count() AS total_records,\n",
    "    100.0 * (1 - (null_count + duplicate_count) / total_records) AS quality_score\n",
    "FROM bronze.snapshot_raw\n",
    "WHERE ref_date = toDate('{ref_date}')\n",
    "GROUP BY table_name\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(quality_metrics_query)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "quality_metrics_result = client.execute_query_with_result(\n",
    "    f\"\"\"\n",
    "    SELECT *\n",
    "    FROM gold.data_quality_metrics\n",
    "    WHERE ref_date = toDate('{ref_date}')\n",
    "    \"\"\"\n",
    ")\n",
    "quality_metrics_result.result_rows"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "historical_trends = client.execute_query_with_result(\n",
    "    \"\"\"\n",
    "    SELECT\n",
    "        ref_date,\n",
    "        table_name,\n",
    "        total_inserts,\n",
    "        total_updates,\n",
    "        total_deletes,\n",
    "        total_active_records\n",
    "    FROM gold.daily_change_metrics\n",
    "    ORDER BY ref_date DESC, table_name\n",
    "    LIMIT 50\n",
    "    \"\"\"\n",
    ")\n",
    "historical_trends.result_rows"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client.close()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.14"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}